In [2]:
import pandas as pd
import re
import nltk
import os
import sys
sys.path.append(os.path.abspath(".."))

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from src.datapreprocessing import load_data,preprocess_text,identify_theme,save_dataset


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bemnet\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Thematic Analysis

## Download NLTK Stopwords

In [3]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bemnet\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Load Sentiment-enhanced Review Dataset.

In [4]:
df = load_data("../data/processed/bank_reviews_sentiment.csv")
df.head()

,review_id,review,rating,date,bank,source,sentiment,confidence_score
0,9f0c0b92-5087-496f-98e2-60a97f2076d1,very nice,5,2026-05-17,Commercial Bank of Ethiopia,Google Play,Positive,0.999856
1,7b8203d4-e95a-4693-b2ea-3e35a6aa797d,f*k,1,2026-05-17,Commercial Bank of Ethiopia,Google Play,Negative,0.700372
2,60e3a921-2d6b-44a8-bb4b-636af4637b83,The Bank You can Always Rely on!,5,2026-05-17,Commercial Bank of Ethiopia,Google Play,Positive,0.998296
3,92e988a8-485c-4691-90ff-4005e0f0b3b7,Please make the CBE Noor toggle to be optional...,2,2026-05-17,Commercial Bank of Ethiopia,Google Play,Negative,0.998795
4,d7c8285b-d811-4738-ba6e-98014135b5a3,Amazing App,5,2026-05-17,Commercial Bank of Ethiopia,Google Play,Positive,0.999867


## Text Preprocessing

In [5]:

df["clean_review"] = df["review"].apply(preprocess_text)

print(df[["review", "clean_review"]].head())
df


                                              review  \
0                                          very nice   
1                                                f*k   
2                   The Bank You can Always Rely on!   
3  Please make the CBE Noor toggle to be optional...   
4                                        Amazing App   

                                        clean_review  
0                                               nice  
1                                                 fk  
2                                   bank always rely  
3  please make cbe noor toggle optional another a...  
4                                        amazing app  


,review_id,review,rating,date,bank,source,sentiment,confidence_score,clean_review
0,9f0c0b92-5087-496f-98e2-60a97f2076d1,very nice,5,2026-05-17,Commercial Bank of Ethiopia,Google Play,Positive,0.999856,nice
1,7b8203d4-e95a-4693-b2ea-3e35a6aa797d,f*k,1,2026-05-17,Commercial Bank of Ethiopia,Google Play,Negative,0.700372,fk
2,60e3a921-2d6b-44a8-bb4b-636af4637b83,The Bank You can Always Rely on!,5,2026-05-17,Commercial Bank of Ethiopia,Google Play,Positive,0.998296,bank always rely
3,92e988a8-485c-4691-90ff-4005e0f0b3b7,Please make the CBE Noor toggle to be optional...,2,2026-05-17,Commercial Bank of Ethiopia,Google Play,Negative,0.998795,please make cbe noor toggle optional another a...
4,d7c8285b-d811-4738-ba6e-98014135b5a3,Amazing App,5,2026-05-17,Commercial Bank of Ethiopia,Google Play,Positive,0.999867,amazing app
...,...,...,...,...,...,...,...,...,...
1195,191e81ad-2e0f-4933-8ae3-214036c68bb7,Best,5,2023-05-04,Dashen Bank,Google Play,Positive,0.999794,best
1196,382555e0-fd93-4ce6-bbb1-9d3579b337fb,Best app,5,2023-04-28,Dashen Bank,Google Play,Positive,0.999687,best app
1197,ece57793-b66d-4559-853d-7520996b7d0e,This app is not expected from a bank that says...,3,2023-04-28,Dashen Bank,Google Play,Negative,0.999660,app expected bank says pioneer adapting techno...
1198,b8ba07f5-9be0-46db-99b3-1a6e49e052cc,Good,5,2023-04-18,Dashen Bank,Google Play,Positive,0.999816,good


## Extract Keywords Using TF-IDF

In [6]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),   # unigrams + bigrams
    max_features=100
)

X = vectorizer.fit_transform(df["clean_review"])

In [7]:
keywords = vectorizer.get_feature_names_out()

scores = X.mean(axis=0).A1

tfidf_df = pd.DataFrame({
    "keyword": keywords,
    "score": scores
})

tfidf_df = tfidf_df.sort_values(
    by="score",
    ascending=False
)

print("\nTop TF-IDF Keywords:")
print(tfidf_df.head(20))
tfidf_df["keyword"].to_csv("full_output.csv", index=False)


Top TF-IDF Keywords:
        keyword     score
37         good  0.165236
5           app  0.110428
15         best  0.065800
57         nice  0.058644
11         bank  0.029337
97      working  0.022248
89          use  0.021887
38     good app  0.021561
16     best app  0.020512
45         like  0.019332
34         fast  0.018062
60           ok  0.017996
88       update  0.017975
96         work  0.016957
32    excellent  0.016804
7   application  0.016566
12      banking  0.015957
58     nice app  0.015464
76      service  0.015242
50       mobile  0.014116


## Bank-Level Keyword Analysis

In [8]:
banks = df["bank"].unique()

for bank in banks:

    print(f"\n{'='*50}")
    print(f"Top Keywords for {bank}")
    print(f"{'='*50}")

    bank_reviews = df[
        df["bank"] == bank
    ]["clean_review"]

    vectorizer = TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),
        max_features=20
    )

    X_bank = vectorizer.fit_transform(bank_reviews)

    keywords_bank = vectorizer.get_feature_names_out()

    scores_bank = X_bank.mean(axis=0).A1

    bank_tfidf = pd.DataFrame({
        "keyword": keywords_bank,
        "score": scores_bank
    })

    bank_tfidf = bank_tfidf.sort_values(
        by="score",
        ascending=False
    )

    print(bank_tfidf)



Top Keywords for Commercial Bank of Ethiopia
        keyword     score
8          good  0.215718
0           app  0.140766
3          best  0.070127
11         nice  0.066692
13           ok  0.042500
4           cbe  0.034199
16       update  0.032369
19      working  0.031763
2          bank  0.031740
1   application  0.028543
17          use  0.027070
9          like  0.027007
14      service  0.024201
15     transfer  0.023194
6     excellent  0.022819
18         work  0.020655
7          fast  0.020596
5          easy  0.019933
10       mobile  0.017895
12     nice app  0.016560

Top Keywords for Bank of Abyssinia
           keyword     score
0              app  0.180766
9             good  0.170300
4             best  0.066069
12            nice  0.050116
1             bank  0.049042
5              boa  0.038278
18         working  0.035328
2          banking  0.028643
10          mobile  0.027980
7             fast  0.027141
17            work  0.025799
16             use  0.02

## Group Keywords into Themes

| Theme                                       | Related Keywords                                                                                                      | Business Meaning                                                                                   |
| ------------------------------------------- | --------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------- |
| **User Satisfaction & Positive Experience** | good, best, nice, excellent, amazing, wow, love, happy, great, good job, app good, easy, easy use, secure, secured    | Users expressing satisfaction with the banking applications, usability, and service quality.       |
| **App Performance & Technical Issues**      | slow, doesnt work, error, keeps, poor, issue, problem, bad, worst, open, fix, tried, frequently, version, android     | Technical problems affecting reliability, speed, and app stability.                                |
| **Transaction & Banking Services**          | transfer, transaction, transactions, payment, money, balance, receipt, send, banking, mobile banking, amole, telebirr | Feedback related to transfers, payments, transaction processing, and digital banking services.     |
| **Account Access & Security**               | login, password, access, account, security, secured, phone, number                                                    | Issues or concerns related to authentication, account access, and security management.             |
| **Feature Improvement & User Requests**     | update, improve, need, needs, option, change, better, new, mobile app, application                                    | User suggestions and requests for application enhancements and new features.                       |



## Theme Grouping Logic

Business-related themes were assigned using a rule-based keyword matching approach. Each review was converted to lowercase and checked against predefined keyword groups representing common customer concerns and discussion areas.

The grouping process followed the logic below:

```text
Review Text
      ↓
Convert Text to Lowercase
      ↓
Keyword Matching
      ↓
Assign Business Theme

In [9]:
df["identified_theme"] = df[
    "clean_review"
].apply(identify_theme)
df.head(100)
df = df.rename(columns={
    "review": "review_text",
    "sentiment": "sentiment_label",
    "confidence_score": "sentiment_score"
})

In [11]:
save_dataset(df,"bank_reviews_theme.csv")

bank_reviews_theme.csv

Dataset saved as bank_reviews_theme.csv


## Topic modeling Using NMF

In [15]:
num_topics = 5

nmf_model = NMF(
    n_components=num_topics,
    random_state=42
)

nmf_model.fit(X)

,"n_components n_components: int or {'auto'} or None, default='auto'Number of components. If `None`, all features are kept.If `n_components='auto'`, the number of components is automatically inferredfrom W or H shapes... versionchanged:: 1.4 Added `'auto'` value... versionchanged:: 1.6 Default value changed from `None` to `'auto'`.",5
,"init init: {'random', 'nndsvd', 'nndsvda', 'nndsvdar', 'custom'}, default=NoneMethod used to initialize the procedure.Valid options:- `None`: 'nndsvda' if n_components <= min(n_samples, n_features), otherwise random.- `'random'`: non-negative random matrices, scaled with: `sqrt(X.mean() / n_components)`- `'nndsvd'`: Nonnegative Double Singular Value Decomposition (NNDSVD) initialization (better for sparseness)- `'nndsvda'`: NNDSVD with zeros filled with the average of X (better when sparsity is not desired)- `'nndsvdar'` NNDSVD with zeros filled with small random values (generally faster, less accurate alternative to NNDSVDa for when sparsity is not desired)- `'custom'`: Use custom matrices `W` and `H` which must both be provided... versionchanged:: 1.1 When `init=None` and n_components is less than n_samples and n_features defaults to `nndsvda` instead of `nndsvd`.",None
,"solver solver: {'cd', 'mu'}, default='cd'Numerical solver to use:- 'cd' is a Coordinate Descent solver.- 'mu' is a Multiplicative Update solver... versionadded:: 0.17 Coordinate Descent solver... versionadded:: 0.19 Multiplicative Update solver.",'cd'
,"beta_loss beta_loss: float or {'frobenius', 'kullback-leibler', 'itakura-saito'}, default='frobenius'Beta divergence to be minimized, measuring the distance between Xand the dot product WH. Note that values different from 'frobenius'(or 2) and 'kullback-leibler' (or 1) lead to significantly slowerfits. Note that for beta_loss <= 0 (or 'itakura-saito'), the inputmatrix X cannot contain zeros. Used only in 'mu' solver... versionadded:: 0.19",'frobenius'
,"tol tol: float, default=1e-4Tolerance of the stopping condition.",0.0001
,"max_iter max_iter: int, default=200Maximum number of iterations before timing out.",200
,"random_state random_state: int, RandomState instance or None, default=NoneUsed for initialisation (when ``init`` == 'nndsvdar' or'random'), and in Coordinate Descent. Pass an int for reproducibleresults across multiple function calls.See :term:`Glossary `.",42
,"alpha_W alpha_W: float, default=0.0Constant that multiplies the regularization terms of `W`. Set it to zero(default) to have no regularization on `W`... versionadded:: 1.0",0.0
,"alpha_H alpha_H: float or ""same"", default=""same""Constant that multiplies the regularization terms of `H`. Set it to zero tohave no regularization on `H`. If ""same"" (default), it takes the same value as`alpha_W`... versionadded:: 1.0",'same'
,"l1_ratio l1_ratio: float, default=0.0The regularization mixing parameter, with 0 <= l1_ratio <= 1.For l1_ratio = 0 the penalty is an elementwise L2 penalty(aka Frobenius Norm).For l1_ratio = 1 it is an elementwise L1 penalty.For 0 < l1_ratio < 1, the penalty is a combination of L1 and L2... versionadded:: 0.17 Regularization parameter *l1_ratio* used in the Coordinate Descent solver.",0.0
,"verbose verbose: int, default=0Whether to be verbose.",0


In [16]:
# feature_names = vectorizer.get_feature_names_out()
# print(feature_names)
for topic_idx, topic in enumerate(nmf_model.components_):

    print(f"\nTopic {topic_idx + 1}")

    top_keywords = [
        keywords[i]
        for i in topic.argsort()[:-11:-1]
    ]

    print(top_keywords)


Topic 1
['good', 'good app', 'application', 'service', 'app good', 'update', 'work', 'use', 'easy', 'need']

Topic 2
['app', 'good app', 'best app', 'nice app', 'amazing', 'time', 'great', 'like', 'app good', 'fast']

Topic 3
['best', 'best app', 'ethiopia', 'used', 'experience', 'bank', 'password', 'boa', 'apps', 'mobile']

Topic 4
['nice', 'nice app', 'application', 'use', 'easy', 'fast', 'problem', 'service', 'wow', 'banking']

Topic 5
['bank', 'working', 'use', 'banking', 'update', 'mobile', 'mobile banking', 'dashen', 'work', 'dashen bank']


| Theme                                   | Related Keywords                                                                    | Business Meaning                                                                                                                   |
| --------------------------------------- | ----------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
| **Service Quality & User Satisfaction** | good, good app, service, application, good job, job, app good, work, use, excellent | Users expressing positive experiences, satisfaction with banking services, and appreciation for app usability and reliability.     |
| **User Experience & App Perception**    | app, best app, nice app, amazing, great, like, app good, worst, time                | Customer opinions regarding overall app quality, usability, and user experience, including both positive and negative perceptions. |
| **Security & Banking Trust**            | password, secured, bank, experience, boa, ethiopia, apps                            | Feedback related to account security, trust in banking services, and user confidence in mobile banking platforms.                  |
| **App Performance & Ease of Use**       | nice, fast, easy, problem, service, try, use, application                           | Discussions about app speed, ease of navigation, convenience, and occasional usability or technical issues.                        |
| **Mobile Banking Functionality**        | bank, banking, mobile banking, mobile, working, update, dashen, work                | Reviews related to core banking features, mobile banking functionality, updates, and operational performance of the applications.  |
